# Creating Raw data and save as delta table.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType
)

In [0]:

# Sample source data

orders_data = [
    (101, "C01", "P01", "2026-10-01", 2, "Completed"),
    (102, "C02", "P02", "2026-10-02", 1, "Completed"),
    (103, "C01", "P03", "2026-10-03", 3, "Pending"),
    (104, "C03", "P01", "2026-10-04", 1, "Cancelled")
]


customers_data = [
    ("C01", "Ravi", "South"),
    ("C02", "Priya", "East"),
    ("C03", "Amit", "West")
]

products_data = [
    ("P01", "Laptop", "Electronics", 50000.0),
    ("P02", "Headphones", "Accessories", 2000.0),
    ("P03", "Keyboard", "Accessories", 1500.0)
]

regions_data = [
    ("South", "Kiran"),
    ("East", "Meena"),
    ("West", "Arjun")
]


In [0]:
bad_orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("status", StringType(), True)
])

In [0]:
orders_data_bad_records = [
    (101, "C01", "P01", "2026-10-01", 2, "Completed"),
    (102, "C02", "P02", "2026-10-02", 1, "Completed"),
    (103, "C01", "P03", "2026-10-03", 3, "Pending"),
    (104, "C03", "P01", "2026-10-04", 1, "Cancelled"),
    (105, "", "P03", "2026-10-03", 3, "Pending"),
    (106, "C03", "", "2026-10-04", 1, "Cancelled"),
    (107, "C01", "P03", "", 3, "Pending"),
    (108, "C03", "P01", "2026-10-04", 1, ""),
    (109, "C03", "", "2026-10-04", 0, "")
]




In [0]:
bad_rec_orders_df=spark.createDataFrame(orders_data_bad_records, bad_orders_schema)

In [0]:

# Bad record table
bad_rec_orders_df.write.format("delta").mode("overwrite").saveAsTable("onehouse_dev_dtu_bronze.source_raw_schema.bad_records_bronze_orders")

In [0]:
%sql
select * from onehouse_dev_dtu_bronze.source_raw_schema.bad_records_bronze_orders

In [0]:
# Define schemas explicitly instead of inferring them
orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("status", StringType(), True)
])

customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), True),
    StructField("region", StringType(), True)
])

products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("unit_price", DoubleType(), True)
])

regions_schema = StructType([
    StructField("region", StringType(), False),
    StructField("region_manager", StringType(), True)
])

In [0]:
# Create DataFrames
orders_df = spark.createDataFrame(orders_data, orders_schema)
customers_df = spark.createDataFrame(customers_data, customers_schema)
products_df = spark.createDataFrame(products_data, products_schema)
regions_df = spark.createDataFrame(regions_data, regions_schema)



# Save as Bronze Delta tables
orders_df.write.format("delta").mode("overwrite").saveAsTable("onehouse_dev_dtu_bronze.source_raw_schema.bronze_orders")
customers_df.write.format("delta").mode("overwrite").saveAsTable("onehouse_dev_dtu_bronze.source_raw_schema.bronze_customers")
products_df.write.format("delta").mode("overwrite").saveAsTable("onehouse_dev_dtu_bronze.source_raw_schema.bronze_products")
regions_df.write.format("delta").mode("overwrite").saveAsTable("onehouse_dev_dtu_bronze.source_raw_schema.bronze_regions")

print("All 4 Bronze tables created successfully")

In [0]:
%sql
show tables in onehouse_dev_dtu_bronze.source_raw_schema;

In [0]:
%sql
select * from onehouse_dev_dtu_bronze.source_raw_schema.bronze_orders

In [0]:
%sql
select * from onehouse_dev_dtu_bronze.source_raw_schema.bronze_customers

In [0]:
%sql
select * from onehouse_dev_dtu_bronze.source_raw_schema.bronze_products

In [0]:
%sql
select * from onehouse_dev_dtu_bronze.source_raw_schema.bronze_regions